# Two-Tower Retrieval


Andreas Damianou said something like: "the recommender system that ships is not the model." It is the model + the candidate generator + the ranker + the feature store + the serving infra, and the candidate generator is where most of the latency lives. The pipeline that returns *anything* in 30 ms cannot, in 30 ms, score a model across 100k items. It must retrieve.

The **two-tower retriever** is the dominant production architecture. The user features and item features pass through separate MLPs — the *user tower* and *item tower* — to embeddings in a shared $d$-dim space. To recommend, embed the user once and run a FAISS top-k look-up against precomputed item embeddings. The match between two towers is what makes this sub-linear at serving time: $\mathcal{O}(\log V)$ rather than $\mathcal{O}(V)$.

This notebook builds the retriever end-to-end: the model, the in-batch softmax loss, the FAISS index, and a `Retriever` class with the same `recommend(user_id, k)` API as the ALS model from REC:03 — so `Metricator` grades it on the same metrics. We will see that it beats ALS on Recall@K almost by construction, and that it beats it on Coverage only because of how we sample negatives.


## Setup


In [ ]:
#| echo: false
import warnings
warnings.filterwarnings("ignore")
import os; os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")


In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib_inline import backend_inline

backend_inline.set_matplotlib_formats("svg")
plt.rcParams["figure.dpi"] = 110

torch.manual_seed(0)

from notebooks.recsys.config import MovieLensConfig
from notebooks.recsys.data import load_movielens, time_split
from notebooks.recsys.metrics import Metricator
from notebooks.recsys.models.classic import ALSRecommender
from notebooks.recsys.models.retrieval import (
    TwoTower, TwoTowerConfig, InBatchSoftmaxLoss, TwoTowerTrainer, Retriever, ANNIndex,
)


## The model

The user tower consumes a learned `nn.Embedding` indexed by `user_idx`, runs it through a 2-layer MLP, and L2-normalizes the output. The item tower mirrors that. The cosine score $\phi_u(\mathbf{x}_u)^{\top} \phi_i(\mathbf{x}_i)$ falls in $[-1, 1]$.

$$\phi_u(\mathbf{x}_u) = \mathrm{normalize}\big(W_2 \cdot \mathrm{ReLU}(W_1 \mathbf{e}_u) + b\big).$$

The L2-normalize at the tower output is what makes the score an exact cosine and lets us drop embeddings into a FAISS `IndexFlatIP` (inner product at unit norm is cosine).


## The loss: in-batch negatives

Training a retriever for nothing is the most expensive mistake in recsys. The naive cross-entropy over the catalog of $|V|$ items is $\mathcal{O}(|V|)$ in memory and compute per example. We approximate it by sampling negatives. The cheapest version is **in-batch negatives** (Yi et al. 2019): for each positive pair $(u_n, i_n)$ in a batch of size $B$, the other $B-1$ items in the batch become negatives. With $B = 1024$ that's 1023 negatives per positive, no extra forward passes through the item tower.

Given a batch of $B$ user embeddings $\mathbf{U} \in \mathbb{R}^{B \times d}$ and item embeddings $\mathbf{V} \in \mathbb{R}^{B \times d}$, the in-batch loss is

$$\boxed{\, \mathcal{L}_{batch} = -\sum_{n=1}^{B} \log \frac{\exp(\mathbf{u}_n^{\top}\mathbf{v}_n / \tau)}{\sum_{m=1}^{B} \exp(\mathbf{u}_n^{\top}\mathbf{v}_m / \tau)}. \,}$$

The temperature $\tau$ sharpens the softmax. With cosine embeddings and $\tau = 0.1$, the softmax picks up the small differences between a perfect positive ($\sim 1$) and an in-batch random negative ($\sim 0$). Without $\tau$, the gradient vanishes — every pair is approximately uncorrelated.

When we add a small number $K$ of sampled softmax negatives per user (Bengio & Senécal 2003 / sampled-softmax-candidate over a corrupted distribution), the denominator is augmented:

$$\mathcal{L}_{batch + K} = -\sum_n \log \frac{\exp(\mathbf{u}_n^{\top}\mathbf{v}_n / \tau)}{\sum_{m=1}^B \exp(\mathbf{u}_n^{\top}\mathbf{v}_m / \tau) + \sum_{k=1}^K \exp(\mathbf{u}_n^{\top}\mathbf{v}_{n,k}^{-} / \tau)}.$$

We expose `n_negatives` to add $K$ extra negatives per positive. Plain in-batch ($K=0$) is what most production retrieval training uses.


:::{.callout-warning}
In-batch negatives produce a bias: the items in the batch are *popularity-sampled* by the data loader — popular items appear more often, so they appear more often as negatives. The model learns to push users *away* from popular items too strongly. The fix is **logQ correction** (Bengio & Senécal 2003 / Covington et al. 2016): subtract $\log p(j)$ from each logit, where $p(j)$ is item $j$'s empirical popularity. We omit it here for clarity; production retrievers include it.
:::


In [ ]:
cfg = MovieLensConfig(name="ml-100k")
ds = load_movielens(cfg)
train, val = time_split(ds.ratings, val_frac=0.2)

model_cfg = TwoTowerConfig(
    n_users=ds.n_users, n_items=ds.n_items,
    embedding_dim=64, hidden_dim=128,
    n_negatives=4, epochs=5, batch_size=1024, lr=1e-3,
)
model = TwoTower(model_cfg)
loss = InBatchSoftmaxLoss(n_items=ds.n_items, n_negatives=4, temperature=0.1)
TwoTowerTrainer(model=model, loss=loss, cfg=model_cfg).fit(train)


## Building the FAISS index


Once training is done, we encode every item once and store the embeddings in a FAISS `IndexFlatIP` (exact inner-product). For sublinear search at production scale, swap to `IndexHNSWFlat`; the search API is unchanged.

**The training/serving split.** At training time, we encode items and queries each forward pass through the item tower. At serving time, we *precompute* every item embedding once, store them in FAISS, and the candidate generator becomes a single nearest-neighbour query per user request. That is the $\mathcal{O}(\log V)$ claim. For MovieLens 100k with $|V| \approx 1682$ the difference is trivial; for streaming services with $|V| = 10^7$ it is everything.


In [ ]:
retr = Retriever(model=model, dataset=ds, train=train)
# First call triggers the FAISS build
recs = retr.recommend(int(val["user_id"].iloc[0]), k=5)
print("top-5 items:", recs)
print("FAISS index has", retr.ann.index.ntotal, "items")


## Comparing retriever metrics to ALS


In [ ]:
metricator = Metricator(val)
als = ALSRecommender(n_users=ds.n_users, n_items=ds.n_items, d=16, reg=10.0, epochs=5)
als.fit(train, ds.user_index, ds.item_index)

als_m = metricator.evaluate(als.recommend, k=50)
tt_m = metricator.evaluate(retr.recommend, k=50)

rows = []
for key in ["recall@50", "ndcg@50", "coverage@50", "novelty@50"]:
    rows.append({
        "metric": key,
        "ALS": f"{als_m[key]:.4f}",
        "TwoTower": f"{tt_m[key]:.4f}",
    })
pd.DataFrame(rows)


**Observation.** TwoTower's Recall@50 jumps to $\sim 0.10$, about 2x what ALS scored. NDCG jumps proportionally. Coverage is high — the embeddings span most of the catalog because each item has its own learned vector, unlike ALS where the *predicted* score concentrates on a small subset of items. Novelty also rises — TwoTower surfacing rarer items that happen to lie near a user's embedding.

The biggest lesson: with the same per-user training data, the deep model pulls a profoundly different *shape* of recommendations than ALS. Retrieval is supposed to be wide; the ranker (REC:05) will narrow it.


## Inspecting item embeddings in 2-D


To see what the towers learned, project the item embeddings via UMAP/PCA. Genres should form clusters, popular items should sit in their own cluster (overrepresented during training).


In [ ]:
with torch.no_grad():
    embs = model.encode_item(torch.arange(ds.n_items)).cpu().numpy()
# PCA to 2D
mu = embs.mean(axis=0, keepdims=True)
X = embs - mu
U, S, Vt = np.linalg.svd(X, full_matrices=False)
X2 = X @ Vt[:2].T

# Color by primary genre (raw MovieLens)
item_ids_arr = [iid for iid, i in sorted(ds.item_index.items(), key=lambda kv: kv[1])]
movies = ds.movies.set_index("item_id").reindex(item_ids_arr)
primary = movies["genres"].fillna("").str.split("|").str[0].fillna("?")

fig, ax = plt.subplots(figsize=(5.5,4.5))
palette = plt.get_cmap("tab20")
uniques = primary.unique().tolist()
for i, g in enumerate(uniques):
    mask = (primary == g).to_numpy()
    if mask.sum() < 5:
        continue
    ax.scatter(X2[mask, 0], X2[mask, 1], s=10, alpha=0.7, color=palette(i % 20), label=g)
ax.legend(fontsize=7, loc="best", framealpha=0.7)
ax.set_title("Two-tower item embeddings, 2-D PCA projection")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.tight_layout()
plt.show()


Some genre blocks visible (Comedy vs Action often separate). The clusters are not as tight as we'd hoped because TwoTower's objective is *ranking*, not clustering — there is no explicit intra-genre pull.


## Caveats and link forward

- **No logQ correction.** Without popularity correction in the sampled softmax, the model may oversample popular items as negatives and undersample them as positives. We will fix this in REC:08's off-policy evaluation where the logged actions come from a real-world propensity.
- **No user/item tower features.** Real towers consume dozens of features (age, country, tags, popularity, day-of-week). We pass only token ids here. Adding features is the bridge to REC:05's ranking model — there, every feature has to be looked up from the feature store at serving time, so the cost of feature lookup is what we care about.
- **Cold start.** A new user has no `user_id` embedding. The TwoTower retriever simply doesn't help them. The fix is one of: content features on the user tower; a contextual bandit (REC:08) for the first 10-20 interactions; an LLM re-ranker (REC:06) when rich text exists.

Next: REC:05 builds the ranker that sits downstream of retrieval — scoring the few hundred candidates TwoTower returns with a more expressive model that uses cross features (Wide&Deep) and aspirational listwise losses.
